In [ ]:
import json
import shutil
import subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
from pathlib import Path

import numpy as np
import polars as pl
from Bio import SeqIO
from ete4 import Tree

In [ ]:
CWD = Path().resolve()
BASE = CWD.parent

print(f"Current working directory: {CWD}")
print(f"Folder where groups (data) folder should be: {BASE}")

In [ ]:
groups_map = {
    "plants_plt": ["solanaceae"],
}

# gene-pair trees to calibrate: <subgroup>/gene_pair_trees/<TREE_RUN_NAME>/gp_{gp}.nwk
TREE_RUN_NAME = "cds_first_pass"
# IGR alignments to fit the scale on: <subgroup>/alignments/<ALN_RUN_NAME>/<ALN_PATTERN>
# Clustal Omega does not use the tree, so the estimate is independent of what is being calibrated.
ALN_RUN_NAME = "clustalo_2026-10-08"
ALN_PATTERN = "gp_{gp}_aligned.fasta"

IQTREE = "iqtree3"
MODEL = "GTR+F+G4"  # one fixed model for every gene pair, so the scales are comparable
WORKERS = 16  # IQ-TREE runs in parallel, one thread each
SEED = 12345
MIN_VARIABLE_SITES = 10  # gene pairs with fewer variable columns are kept in the table but left out of the summary
OVERWRITE = False  # True reruns every fit and allows changing the settings of an existing estimate

if shutil.which(IQTREE) is None:
    raise FileNotFoundError(f"'{IQTREE}' not found in PATH")
iqtree_version = subprocess.run([IQTREE, "--version"], capture_output=True, text=True).stdout.splitlines()[0].strip()
print(f"IQ-TREE: {iqtree_version}")

In [ ]:
def check_run_settings(run_json: Path, settings: dict) -> None:
    """Stop if this estimate already exists with different settings (unless OVERWRITE)."""
    if not run_json.exists():
        return
    old = json.loads(run_json.read_text())
    changed = {k: (old.get(k), v) for k, v in settings.items() if old.get(k) != v}
    if changed and not OVERWRITE:
        details = "; ".join(f"{k}: {a!r} -> {b!r}" for k, (a, b) in changed.items())
        raise RuntimeError(
            f"{run_json.parent} already exists with different settings ({details}). "
            "Set OVERWRITE = True to replace it."
        )


def tree_length(newick: str) -> float:
    return sum(n.dist or 0.0 for n in Tree(newick, parser=5).traverse())


def alignment_stats(records: list) -> dict:
    """Number of sequences, length, variable columns (>= 2 different bases, gaps ignored), gap %."""
    mat = np.array([list(str(r.seq).upper()) for r in records], dtype="U1")
    gaps = (mat == "-") | (mat == "N")
    variable = 0
    for col, gap_col in zip(mat.T, gaps.T):
        if len(set(col[~gap_col])) > 1:
            variable += 1
    return {
        "n_seqs": mat.shape[0],
        "aln_len": mat.shape[1],
        "variable_sites": variable,
        "gap_pct": round(100 * float(gaps.mean()), 2),
    }


def fit_scale(gp, aln: Path, tree_file: Path, out_dir: Path) -> dict:
    """Fit one scaling factor for the fixed tree (topology and relative branch lengths kept)."""
    prefix = out_dir / f"gp_{gp}"
    treefile = Path(f"{prefix}.treefile")
    if not (treefile.exists() and not OVERWRITE):
        cmd = [
            IQTREE, "-s", str(aln), "-te", str(tree_file), "-blscale",
            "-m", MODEL, "-keep-ident", "-T", "1", "--seed", str(SEED),
            "--prefix", str(prefix), "--quiet", "-redo",
        ]
        res = subprocess.run(cmd, capture_output=True, text=True)
        if res.returncode != 0 or not treefile.exists():
            Path(f"{prefix}.error.log").write_text(res.stdout + res.stderr)
            return {"gene_pair": gp, "status": f"iqtree_error (exit {res.returncode})", "scale": None}
    scale = tree_length(treefile.read_text()) / tree_length(tree_file.read_text())
    return {"gene_pair": gp, "status": "ok", "scale": scale}


def weighted_median(values: np.ndarray, weights: np.ndarray) -> float:
    order = np.argsort(values)
    cum = np.cumsum(weights[order])
    return float(values[order][np.searchsorted(cum, cum[-1] / 2)])

In [ ]:
for g, subgs in groups_map.items():
    for subg in subgs:
        subg_dir = BASE / g / subg
        tree_run_dir = subg_dir / "igr_trees" / TREE_RUN_NAME
        aln_dir = subg_dir / "alignments" / ALN_RUN_NAME
        out_dir = tree_run_dir / "scale" / ALN_RUN_NAME
        out_dir.mkdir(parents=True, exist_ok=True)
        run_json = out_dir / "run.json"
        print(f"Processing {g}/{subg} | trees: {TREE_RUN_NAME} | alignments: {ALN_RUN_NAME}")

        tree_info = json.loads((tree_run_dir / "run.json").read_text())
        input_scale = float(tree_info.get("scale", 1.0))

        settings = {
            "tree_run_name": TREE_RUN_NAME,
            "aln_run_name": ALN_RUN_NAME,
            "model": MODEL,
            "seed": SEED,
            "min_variable_sites": MIN_VARIABLE_SITES,
        }
        check_run_settings(run_json, settings)

        # --- pass 1: every gene pair with a tree needs an alignment with exactly the same IGR IDs
        tree_files = sorted(tree_run_dir.glob("gp_*.nwk"))
        jobs, stats, problems = [], {}, []
        for tree_file in tree_files:
            gp = tree_file.stem.removeprefix("gp_")
            aln = aln_dir / ALN_PATTERN.format(gp=gp)
            if not aln.exists():
                problems.append(f"{gp}: missing {aln.name}")
                continue
            records = list(SeqIO.parse(aln, "fasta"))
            tips = {leaf.name for leaf in Tree(tree_file.read_text(), parser=5).leaves()}
            ids = {r.id for r in records}
            if ids != tips:
                problems.append(f"{gp}: alignment and tree differ ({len(ids - tips)} only in alignment, {len(tips - ids)} only in tree)")
                continue
            stats[gp] = alignment_stats(records)
            jobs.append((gp, aln, tree_file))

        if problems:
            print(f"  {len(problems)} of {len(tree_files)} gene pairs have problems:")
            for p in problems:
                print(f"    {p}")
            raise RuntimeError(f"{g}/{subg}: alignments do not match the gene-pair trees; nothing was fitted.")
        print(f"  {len(jobs)} gene pairs | fitting one scale each with {MODEL} ({WORKERS} in parallel)")

        run_info = {
            "status": "running",
            "date_started": f"{datetime.now():%Y-%m-%d %H:%M:%S}",
            **settings,
            "group": g,
            "subgroup": subg,
            "input_tree_scale": input_scale,
            "tree_run_dir": str(tree_run_dir.relative_to(subg_dir)),
            "tree_run_dir_abs": str(tree_run_dir.resolve()),
            "aln_dir": str(aln_dir.relative_to(subg_dir)),
            "aln_dir_abs": str(aln_dir.resolve()),
            "iqtree": IQTREE,
            "iqtree_version": iqtree_version,
        }
        run_json.write_text(json.dumps(run_info, indent=2))

        # --- pass 2: fit (gene pairs with no variable site have nothing to fit)
        rows = []
        to_fit = [(gp, aln, tf) for gp, aln, tf in jobs if stats[gp]["variable_sites"] > 0]
        rows += [{"gene_pair": gp, "status": "no_variation", "scale": None} for gp, _, _ in jobs if stats[gp]["variable_sites"] == 0]
        with ThreadPoolExecutor(max_workers=WORKERS) as executor:
            futures = [executor.submit(fit_scale, gp, aln, tf, out_dir) for gp, aln, tf in to_fit]
            for i, future in enumerate(as_completed(futures), 1):
                rows.append(future.result())
                if i % 25 == 0 or i == len(futures):
                    print(f"    {i}/{len(futures)} fitted")

        est = (
            pl.DataFrame(rows, schema={"gene_pair": pl.String, "status": pl.String, "scale": pl.Float64})
            .join(
                pl.DataFrame([{"gene_pair": gp, **s} for gp, s in stats.items()]),
                on="gene_pair",
                how="left",
            )
            .with_columns(
                ((pl.col("status") == "ok") & (pl.col("variable_sites") >= MIN_VARIABLE_SITES)).alias("used_in_summary")
            )
            .sort("scale", descending=True, nulls_last=True)
        )
        est.write_csv(out_dir / "scale_estimates.tsv", separator="\t")

        # --- summary: scale relative to the input trees, and the SCALE to use in 05
        used = est.filter("used_in_summary")
        vals = used["scale"].to_numpy()
        weights = used["variable_sites"].to_numpy().astype(float)
        q25, med, q75 = np.percentile(vals, [25, 50, 75])
        wmed = weighted_median(vals, weights)
        suggested = input_scale * wmed

        run_info.update(
            status="complete",
            date_finished=f"{datetime.now():%Y-%m-%d %H:%M:%S}",
            n_gene_pairs=len(jobs),
            n_used=used.height,
            median=float(med),
            weighted_median=wmed,
            q25=float(q25),
            q75=float(q75),
            suggested_scale=suggested,
        )
        run_json.write_text(json.dumps(run_info, indent=2))

        print(f"  -> {out_dir.relative_to(subg_dir)}/scale_estimates.tsv, run.json")
        print(est.group_by("status").agg(pl.len().alias("gene_pairs")))
        print(
            f"  used: {used.height}/{len(jobs)} (status ok, >= {MIN_VARIABLE_SITES} variable sites)"
            f" | scale median {med:.2f} [IQR {q25:.2f}-{q75:.2f}] | weighted median (by variable sites) {wmed:.2f}"
            f" | min {vals.min():.2f} | max {vals.max():.2f}"
        )
        print(
            f"  input trees had SCALE = {input_scale}; suggested SCALE for 05: {suggested:.3f}"
            f" (max root-to-tip {tree_info.get('max_root_to_tip', float('nan')) / input_scale * suggested:.4f})"
        )

In [ ]:
# Per gene pair: largest scales first (fast-evolving IGRs), with alignment size and variation
est1 = est.select("gene_pair", "status", "scale", "n_seqs", "aln_len", "variable_sites", "gap_pct", "used_in_summary")

In [ ]:
for row in est1.iter_rows(named=True):
    print(f"{row['gene_pair']}\t{row['status']}\t{row['scale']}\t{row['n_seqs']}\t{row['aln_len']}\t{row['variable_sites']}\t{row['gap_pct']}\t{row['used_in_summary']}")